In [2]:
import json
import pandas as pd
import glob
import os

In [3]:
def extract_ratings(df_reviews, df_meta):
    
    """ 
    Inputs: df_reviews = review data for a given state
            df_meta = meta data for that state

    Output: reviews of restaurants only in that state, along with star rating and type of restaurant
    """
    
    # Separates the categories into separate entries
    df_meta = df_meta.explode('category') 
    
    # Maps data has many types of 'restaurants' with various adjectives
    filtered = df_meta[df_meta['category'].str.contains('estaurant', na=False)] 
    filtered = filtered[filtered['category'] != 'Restaurant supply store'] # this is not a type of restaurant
    # add this too: 'Bar restaurant furniture store'
    
    # Export gmaps ids from restaurants only
    # A dictionary is used here as a lookup table to add the restaurant type to the review data
    gmap_id_dict = pd.Series(filtered['category'].values, index=filtered['gmap_id']).to_dict()
    
    # Throw away reviews not of restaurants
    df_reviews = df_reviews[df_reviews['gmap_id'].isin(gmap_id_dict)].copy()
    
    # Add restaurant type
    df_reviews['type'] = df_reviews['gmap_id'].map(gmap_id_dict)
    
    # Keep only rating, text, and type columns
    columns_to_keep = ['rating', 'text', 'type']
    columns_to_drop = [col for col in df_reviews.columns if col not in columns_to_keep]
    df_reviews.drop(columns=columns_to_drop, inplace=True) 
    
    # Throw away entries without any review text
    df_reviews = df_reviews[df_reviews['text'].astype(bool)]

    return df_reviews


In [14]:
def combine_csv_chunks(folder_path, output_file, chunksize=1000000):
    """Combines CSV files in chunks to handle large files."""

    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not csv_files:
        print("No CSV files found.")
        return

    first_file = True  # To handle header writing

    for file in csv_files:
        try:
            for chunk in pd.read_csv(file, chunksize=chunksize):
                if first_file:
                    chunk.to_csv(output_file, mode='a', header=True, index=False)
                    first_file = False
                else:
                    chunk.to_csv(output_file, mode='a', header=False, index=False)
        except Exception as e:
            print(f"Error processing {file}: {e}")

    if not first_file: #check if any data was written.
        print(f"Combined data saved to {output_file}")
    else:
        print("No data was able to be combined.")

In [5]:
def read_json_lines_chunks(file_path, chunk_size=1000):
    """
    Reads lines-delimited JSON files in chunks using pandas.

    Args:
        file_path (str): The path to the JSON file.
        chunk_size (int): The number of JSON lines to read in each chunk.

    Yields:
        pandas.DataFrame: A DataFrame representing a chunk of the JSON data.
    """
    with open(file_path, 'r') as f:
        while True:
            lines = []
            for _ in range(chunk_size):
                line = f.readline()
                if not line:
                    break  # End of file
                lines.append(line)

            if not lines:
                break  # No more lines to read

            chunk_str = ''.join(lines)  # Combine lines into a single string
            try:
                chunk_df = pd.read_json(io.StringIO(chunk_str), lines=True, orient='columns')
                yield chunk_df
            except ValueError as e:
                print(f"Error parsing JSON chunk: {e}")
                print(f"Problematic chunk:\n{chunk_str}")


In [3]:
# Edit this per location
location = 'east-south-central'

# Select location folder
folder_path = f'./google-local-data/{location}'

# Read all review and metadata files from the given location
review_pattern = os.path.join(folder_path, "review*.json")  
meta_pattern = os.path.join(folder_path, "meta*.json") 

reviews = sorted(glob.glob(review_pattern))
metadata = sorted(glob.glob(meta_pattern))

if len(reviews) != len(metadata):
    print('something wrong')

# Make dataframe to concatenate with
df = pd.DataFrame()

for i in range(len(reviews)):
    ## Keep track of progress
    print(i, reviews[i])

    # Read the reviews and metadata
    meta = pd.read_json(metadata[i], lines=True, orient='columns')

    # Use other version if the file is large
    # this crashes if the json file is >4GB
    review = pd.read_json(reviews[i], lines=True, orient='columns')
    
    # Run review extraction script and add to existing df
    ratings = extract_ratings(review, meta)

    # Save file to ratings folder
    file_name = f'./google-local-data/restaurant-ratings/{location}-{i}.csv'
    ratings.to_csv(file_name, index=False)

0 ./google-local-data/east-south-central/review-Alabama_10.json
1 ./google-local-data/east-south-central/review-Kentucky_10.json
2 ./google-local-data/east-south-central/review-Mississippi_10.json
3 ./google-local-data/east-south-central/review-Tennessee_10.json


In [11]:
# This version deals with large json files

# Edit this per location
location = 'pacific'

# Select location folder
folder_path = f'./google-local-data/{location}'

# Read all review and metadata files from the given location
review_pattern = os.path.join(folder_path, "review*.json")  
meta_pattern = os.path.join(folder_path, "meta*.json") 

reviews = sorted(glob.glob(review_pattern))
metadata = sorted(glob.glob(meta_pattern))

if len(reviews) != len(metadata):
    print('something wrong')

for i in range(len(reviews)):
    ## Keep track of progress
    print(i, reviews[i])

    # Read the reviews and metadata
    meta = pd.read_json(metadata[i], lines=True, orient='columns')

    # Deal with the large json files
    j = 0
    for chunk in read_json_lines_chunks(reviews[i], chunk_size=1000000):
        ratings = extract_ratings(chunk, meta)
    
        # Save file to ratings folder
        file_name = f'./google-local-data/restaurant-ratings/{location}/{i}-{j}.csv'
        ratings.to_csv(file_name, index=False)

        # Use this to keep the names distinct
        j += 1

0 ./google-local-data/pacific/review-Alaska_10.json
1 ./google-local-data/pacific/review-California_10.json
2 ./google-local-data/pacific/review-Hawaii_10.json
3 ./google-local-data/pacific/review-Oregon_10.json
4 ./google-local-data/pacific/review-Washington_10.json


In [16]:
# Combine the pieces of the csv files

combine_csv_chunks(folder_path='./google-local-data/restaurant-ratings/pacific',
                   output_file='./google-local-data/restaurant-ratings/pacific.csv')

combine_csv_chunks(folder_path='./google-local-data/restaurant-ratings/south-atlantic',
                   output_file='./google-local-data/restaurant-ratings/south-atlantic.csv')

combine_csv_chunks(folder_path='./google-local-data/restaurant-ratings/west-south-central',
                   output_file='./google-local-data/restaurant-ratings/west-south-central.csv')

Combined data saved to ./google-local-data/restaurant-ratings/pacific.csv
Combined data saved to ./google-local-data/restaurant-ratings/south-atlantic.csv
Combined data saved to ./google-local-data/restaurant-ratings/west-south-central.csv


In [18]:
df = pd.read_csv('./restaurant-ratings/northeast/new-england.csv')

In [31]:
pd.set_option('display.max_rows', None)
df['type'].value_counts()


type
Restaurant                              1049686
Takeout Restaurant                       192639
Pizza restaurant                         166122
Seafood restaurant                       151175
American restaurant                      146832
Fast food restaurant                     114004
Italian restaurant                        94057
Breakfast restaurant                      85933
Delivery Restaurant                       73116
Family restaurant                         68316
Sushi restaurant                          52137
Vegetarian restaurant                     50566
Mexican restaurant                        49241
Chinese restaurant                        42527
Soup restaurant                           38034
Traditional American restaurant           36515
Hamburger restaurant                      31398
New American restaurant                   28457
New England restaurant                    26385
Lunch restaurant                          22198
Taco restaurant                    